# Training Ulang Struktur URL/QR v7

Mengikuti [protokol sebelum fitting](ai-agent-docs/structural-retraining-v7-plan.md). Classifier tetap Logistic Regression. Audit development memberi hipotesis fitur; Phish360 adalah test sumber baru setelah pemeriksaan exposure. Label publik penulis tidak disebut dua audit independen atau gold Indonesia. Semua benchmark v3-v6 tetap teramati. Tidak membuka target atau menampilkan URL dataset.

Jalankan Run All dengan kernel Python 3.12 dari environment ML. Transformasi berada pada modul structural_features; notebook ini menjalankan fitting, pembekuan dan evaluasi nyata. Bila artifact sudah dibekukan, Run All memverifikasi checksum dan memuat hasil; eksperimen berubah memerlukan ID/protokol baru.

In [1]:
import gc
import hashlib
import io
import json
import sqlite3
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path

import joblib
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from IPython.display import display
from scipy.sparse import csr_matrix, hstack
from scipy.special import logit
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss
from sklearn.preprocessing import normalize
from threadpoolctl import threadpool_limits

from guardian_ml.metadata import checksum, psl_identity
from guardian_ml.paths import ROOT
from guardian_ml.phish360_source import prepare_test_source
from guardian_ml.preprocessing import prepare_url
from guardian_ml.rich_artifact import RichArtifact
from guardian_ml.rich_features import numeric_values as old_numeric
from guardian_ml.rich_features import probabilities as old_probabilities
from guardian_ml.structural_features import (
    SCHEMA_VERSION,
    SPECIFICATION,
    StructuralEncoder,
    structural_values,
    text_matrix,
    probabilities,
    raw_probabilities,
)
from guardian_ml.structural_evaluation import (
    paired_domain_interval,
    safe_metrics,
    selection_diagnostics,
    source_point,
)

SEED = 20261101
RUN = ROOT / "ml/runs/notebook-tfidf-logistic-v7"
REPORTS, DATA, ARTIFACTS = RUN / "reports", RUN / "data", RUN / "artifacts"
for folder in [REPORTS, DATA, ARTIFACTS]:
    folder.mkdir(parents=True, exist_ok=True)
NOTEBOOK = ROOT / "guardian_lab_url_qr_structural_v7.ipynb"
MODEL = ARTIFACTS / "url-core-logistic-v7.joblib"
FROZEN = REPORTS / "frozen-v7.json"
REUSE_FROZEN = FROZEN.exists()
V6 = ROOT / "ml/runs/notebook-tfidf-logistic-v6"
V5 = ROOT / "ml/runs/notebook-tfidf-logistic-v5"
LOCAL = ROOT / "ml/data/structural-v7"
LOCAL.mkdir(parents=True, exist_ok=True)
base_artifact = RichArtifact()
base = base_artifact.bundle
protected = {
    path: checksum(ROOT / path)
    for path in [
        "guardian_lab_url_qr.ipynb",
        "guardian_lab_url_qr_data_v6.ipynb",
        "ml/runs/notebook-tfidf-logistic-v6/artifacts/url-core-logistic-v6.joblib",
        "ml/runs/notebook-tfidf-logistic-v6/reports/frozen-v6.json",
        "ml/policies/shadow-runtime-manifest.json",
    ]
}
CONFIGS = [
    {"name": "repeat_v6", "kind": "repeat_v6", "C": 2.0},
    {"name": "structural_linear_c2", "kind": "linear", "C": 2.0},
    {"name": "structural_spline_c05", "kind": "spline", "C": 0.5},
    {"name": "structural_spline_c2", "kind": "spline", "C": 2.0},
]


def write_json(path, value):
    Path(path).write_text(
        json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False) + "\n",
        encoding="utf-8",
    )


def hash_value(value):
    return hashlib.sha256(
        json.dumps(value, sort_keys=True, ensure_ascii=True).encode()
    ).hexdigest()


def notebook_code_hash():
    document = json.loads(NOTEBOOK.read_text(encoding="utf-8"))
    return hashlib.sha256(
        "\n\n".join(
            "".join(c["source"]) if isinstance(c["source"], list) else c["source"]
            for c in document["cells"]
            if c["cell_type"] == "code"
        ).encode()
    ).hexdigest()


def canonical_frame(frame):
    for row in frame[["core", "digest", "group"]].itertuples(index=False):
        parsed = prepare_url(row.core)
        assert (
            parsed.core == row.core
            and parsed.digest == row.digest
            and parsed.group == row.group
        )
    return True


print("Research experiment v7; model v6 and public application remain unchanged.")

Research experiment v7; model v6 and public application remain unchanged.


## Audit ulang development dan karantina ambiguitas

Label tidak diubah secara otomatis. Seluruh anggota konflik representasi pada fit dikarantina. Hanya kasus ambigu audit publik yang berada di fit dikeluarkan; tune, calibration, dan threshold tetap utuh. TF-IDF v6 digunakan kembali dengan bukti bahwa vocabulary hanya pernah melihat parent fit v6; exposure unsupervised terhadap baris karantina dicatat.

In [2]:
development_path = V6 / "data/development-v6.parquet"
assert checksum(development_path) == base_artifact.metadata["development_sha256"]
development = pd.read_parquet(development_path)
assert len(development) == 703604
assert development.digest.is_unique and set(development.label.unique()) == {0, 1}
assert canonical_frame(development)
partitions = ["fit", "tune", "calibration", "threshold"]
for i, left in enumerate(partitions):
    left_groups = set(development.loc[development.partition.eq(left), "group"])
    for right in partitions[i + 1 :]:
        assert left_groups.isdisjoint(
            development.loc[development.partition.eq(right), "group"]
        )
original_fit = development[development.partition.eq("fit")].reset_index(drop=True)
view = original_fit.core.str.replace(r"^https?://", "", regex=True)
conflicts = set(
    original_fit.assign(view=view)
    .groupby("view")
    .label.nunique()
    .loc[lambda x: x > 1]
    .index
)
conflict_mask = view.isin(conflicts)
proposal_path = ROOT / "ml/data/model-hardening/coordinator/label-change-proposals.json"
proposals = json.loads(proposal_path.read_text())
case_ids = {item["case_id"] for item in proposals["proposals"]}
previous_manifest = json.loads(
    (V5 / "reports/frozen-new-test-manifest.json").read_text()
)
asset = next(
    item
    for item in previous_manifest["source_assets"]
    if item["role"] == "phishvn_human:annotator_A.csv"
)
response = requests.get(asset["source_url"], timeout=(30, 90))
response.raise_for_status()
assert hashlib.sha256(response.content).hexdigest() == asset["sha256"]
review = pd.read_csv(io.BytesIO(response.content), dtype=str, keep_default_na=False)
ambiguous_digests = {
    prepare_url(row["url"]).digest
    for row in review.to_dict("records")
    if row["vid"] in case_ids
}
assert len(ambiguous_digests) == 3
ambiguity_mask = original_fit.digest.isin(ambiguous_digests)
quarantine = original_fit[conflict_mask | ambiguity_mask].copy()
fit = original_fit[~(conflict_mask | ambiguity_mask)].reset_index(drop=True)
tune = development[development.partition.eq("tune")].reset_index(drop=True)
calibration = development[development.partition.eq("calibration")].reset_index(
    drop=True
)
threshold_data = development[development.partition.eq("threshold")].reset_index(
    drop=True
)
assert fit.groupby("training_source").label.nunique().eq(2).all()
fit_path = DATA / "fit-v7.parquet"
if not fit_path.exists():
    fit.to_parquet(fit_path, index=False)
else:
    cached = pd.read_parquet(fit_path)
    assert fit.equals(cached)
fit_audit = {
    "parent_development_sha256": checksum(development_path),
    "parent_fit_rows": len(original_fit),
    "fit_rows": len(fit),
    "conflicting_fit_representations": len(conflicts),
    "conflicting_fit_rows": int(conflict_mask.sum()),
    "published_ambiguous_fit_rows": int(ambiguity_mask.sum()),
    "total_quarantined_fit_rows": len(quarantine),
    "quarantined_digest_sha256": hash_value(sorted(quarantine.digest)),
    "fit_sha256": checksum(fit_path),
    "published_sheet_sha256": asset["sha256"],
    "label_proposals_sha256": checksum(proposal_path),
    "automatic_relabels_applied": 0,
    "unchanged_partition_rows": {
        p: int(development.partition.eq(p).sum()) for p in partitions if p != "fit"
    },
    "fit_source_class_counts": fit.groupby(["training_source", "label"])
    .size()
    .rename("rows")
    .reset_index()
    .to_dict("records"),
    "text_encoder_provenance": "frozen_v6_parent_fit_only_unsupervised_exposure_includes_quarantined_rows",
    "text_encoder_parent_artifact_sha256": base_artifact.manifest["bundle_sha256"],
    "campaign_independence": "unverified",
}
write_json(REPORTS / "development-audit.json", fit_audit)
display(pd.DataFrame(fit_audit["fit_source_class_counts"]))
print(
    "Fit-only quarantine:",
    {
        k: v
        for k, v in fit_audit.items()
        if k.endswith("rows") or "representations" in k
    },
)

,training_source,label,rows
0,deepurlbench,0,89136
1,deepurlbench,1,10864
2,mendeley2020,0,1868
3,mendeley2020,1,2248
4,phishvn_core,0,8212
5,phishvn_core,1,1555
6,phiusiil,0,72042
7,phiusiil,1,44014
8,phresh,0,162080
9,phresh,1,140732


Fit-only quarantine: {'parent_fit_rows': 532793, 'fit_rows': 532751, 'conflicting_fit_representations': 20, 'conflicting_fit_rows': 40, 'published_ambiguous_fit_rows': 2, 'total_quarantined_fit_rows': 42, 'unchanged_partition_rows': {'tune': 84000, 'calibration': 40104, 'threshold': 46707}}


## Test sumber baru Phish360

Sumber: [halaman penulis](https://web.cs.hacettepe.edu.tr/~selman/phish360-dataset/) dan [paper CrossPhire](https://doi.org/10.3390/app16020751). Hanya kolom URL, Class, dan tag bahasa dari dua Parquet yang dibaca melalui HTTP range. Tidak membaca HTML/screenshot atau mengunjungi target. Hash projection/footer dicatat; hash seluruh binary sumber tidak diklaim. Cache awal tersanitasi diperiksa dan dimanifestkan sebelum fitting.

Tier label adalah klaim kurasi manual penulis, tanpa dua lembar review independen per kasus. Sumber historis ini bukan gold set Indonesia; tag bahasa tidak membuktikan sasaran Indonesia.

In [3]:
external_manifest_path = LOCAL / "phish360-manifest.json"
external_path = LOCAL / "phish360-new.parquet"

external_manifest = prepare_test_source(ROOT)
assert (
    checksum(ROOT / external_manifest["output_path"])
    == external_manifest["output_sha256"]
)
assert external_manifest["rows"] == 7516
external = pd.read_parquet(external_path)
assert external.label.value_counts().to_dict() == {0: 4315, 1: 3201}
assert canonical_frame(external) and external.digest.is_unique
assert set(external.label.unique()) == {0, 1}
assert set(external.group).isdisjoint(development.group)
registry = ROOT / "ml/data/indonesia-gold/registry/known-exposure.sqlite"
snapshot_dir = LOCAL / "registry-snapshots"
snapshot_dir.mkdir(exist_ok=True)
snapshot = snapshot_dir / (external_manifest["registry_before_sha256"] + ".sqlite")
if not snapshot.exists():
    assert checksum(registry) == external_manifest["registry_before_sha256"]
    import shutil

    shutil.copyfile(registry, snapshot)
assert checksum(snapshot) == external_manifest["registry_before_sha256"]
with sqlite3.connect("file:" + snapshot.as_posix() + "?mode=ro", uri=True) as db:
    for row in external.itertuples(index=False):
        assert not db.execute(
            "SELECT 1 FROM url_hashes WHERE value=?", (row.digest,)
        ).fetchone()
        assert not db.execute(
            "SELECT 1 FROM domain_hashes WHERE value=?",
            (hashlib.sha256(row.group.encode()).hexdigest(),),
        ).fetchone()
external_run_path = DATA / "phish360-new.parquet"
if not external_run_path.exists():
    external.to_parquet(external_run_path, index=False)
assert checksum(external_run_path) == external_manifest["output_sha256"]
test_manifest = {
    **external_manifest,
    "output_path": external_run_path.relative_to(ROOT).as_posix(),
    "frozen_before_classifier_fitting": True,
    "selection_calibration_threshold_use": False,
    "indonesia_gold": False,
    "language_counts": external.language.value_counts().to_dict(),
}
write_json(REPORTS / "frozen-new-test-manifest.json", test_manifest)
print(
    "Novel test:",
    len(external),
    "rows;",
    external.group.nunique(),
    "domains;",
    external.label.value_counts().to_dict(),
)
print("Label tier:", test_manifest["label_tier"])

Novel test: 7516 rows; 7002 domains; {0: 4315, 1: 3201}
Label tier: published_author_manual_curation_claim_not_two_independent_case_reviews


## Fitting nyata dan pemilihan hanya dari tune

Empat konfigurasi telah ditetapkan di protokol. Source weights mengikuti v6. Seluruh classifier benar-benar di-fit pada fit yang sama; transformasi numerik baru tidak melihat tune. Urutan seleksi: macro recall, minimum source recall, average precision pada anggaran FPR empiris 1% per sumber.

In [4]:
code_hash = notebook_code_hash()
source_hashes = {
    path: checksum(ROOT / path)
    for path in [
        "ml/src/guardian_ml/structural_features.py",
        "ml/src/guardian_ml/structural_evaluation.py",
        "ml/src/guardian_ml/phish360_source.py",
        "ml/src/guardian_ml/rich_features.py",
        "ml/src/guardian_ml/preprocessing.py",
        "ml/src/guardian_ml/features.py",
        "ml/src/guardian_ml/evaluation.py",
    ]
}
specification = {
    "configs": CONFIGS,
    "feature_specification": SPECIFICATION,
    "data_audit": fit_audit,
    "new_test_sha256": checksum(external_run_path),
    "code_sha256": code_hash,
    "source_hashes": source_hashes,
    "seed": SEED,
    "solver": "liblinear",
    "max_iter": 1500,
    "tol": 0.0001,
    "source_weights": "sqrt(max_source_fit_count/source_fit_count)_mean_normalized",
    "versions": base_artifact.metadata["versions"],
    "psl": psl_identity(),
    "selection": "lexicographic_macro_recall_min_source_recall_average_precision",
}
spec_hash = hash_value(specification)
if REUSE_FROZEN:
    frozen = json.loads(FROZEN.read_text())
    assert frozen["spec_sha256"] == spec_hash
    assert checksum(MODEL) == frozen["bundle_sha256"]
    bundle = joblib.load(MODEL)
    validation_results = json.loads((REPORTS / "validation-results.json").read_text())
    selected = frozen["selected"]
else:
    assert not MODEL.exists() and not (REPORTS / "evaluation-v7.json").exists()
    write_json(
        REPORTS / "notebook-protocol.json",
        {
            "declared_at_utc": datetime.now(timezone.utc).isoformat(),
            "spec_sha256": spec_hash,
            "specification": specification,
            "test_predictions_observed": False,
        },
    )
    print("Preparing fit-only feature blocks...", flush=True)
    fit_blocks = structural_values(fit.core)
    tune_blocks = structural_values(tune.core)
    assert np.array_equal(fit_blocks[0][:64], old_numeric(fit.core.iloc[:64]))
    text_fit = text_matrix(base["encoder"], fit.core)
    text_tune = text_matrix(base["encoder"], tune.core)
    counts = fit.training_source.value_counts()
    weights = np.sqrt(
        counts.max() / fit.training_source.map(counts).to_numpy(dtype=float)
    )
    weights /= weights.mean()
    validation_results = []
    for config in CONFIGS:
        print("Fitting", config["name"], "on", len(fit), "URLs...", flush=True)
        structure = None
        if config["kind"] == "repeat_v6":
            matrix_fit = hstack(
                [
                    text_fit,
                    0.2
                    * csr_matrix(
                        normalize(
                            np.clip(
                                base["encoder"]["scaler"].transform(fit_blocks[0]),
                                0,
                                10,
                            )
                        )
                    ),
                ],
                format="csr",
                dtype=np.float32,
            )
            matrix_tune = hstack(
                [
                    text_tune,
                    0.2
                    * csr_matrix(
                        normalize(
                            np.clip(
                                base["encoder"]["scaler"].transform(tune_blocks[0]),
                                0,
                                10,
                            )
                        )
                    ),
                ],
                format="csr",
                dtype=np.float32,
            )
        else:
            structure = StructuralEncoder(config["kind"]).fit(
                fit_blocks, partition="fit"
            )
            matrix_fit = hstack(
                [text_fit, structure.transform(fit_blocks)],
                format="csr",
                dtype=np.float32,
            )
            matrix_tune = hstack(
                [text_tune, structure.transform(tune_blocks)],
                format="csr",
                dtype=np.float32,
            )
        classifier = LogisticRegression(
            C=config["C"],
            solver="liblinear",
            max_iter=1500,
            tol=0.0001,
            random_state=SEED,
        )
        started = time.perf_counter()
        with (
            threadpool_limits(limits=4),
            warnings.catch_warnings(record=True) as caught,
        ):
            warnings.simplefilter("always", ConvergenceWarning)
            classifier.fit(matrix_fit, fit.label.to_numpy(), sample_weight=weights)
            raw = classifier.predict_proba(matrix_tune)[:, 1]
        converged = not any(isinstance(w.message, ConvergenceWarning) for w in caught)
        assert converged, "Candidate did not converge; do not proceed to test."
        candidate = {
            "encoder": base["encoder"],
            "structure": structure,
            "kind": config["kind"],
            "classifier": classifier,
        }
        candidate_path = ARTIFACTS / (config["name"] + ".joblib")
        joblib.dump(candidate, candidate_path, compress=3)
        result = {
            **config,
            "fit_rows": len(fit),
            "features": matrix_fit.shape[1],
            "classifier_fit_called": True,
            "converged": converged,
            "iterations": int(classifier.n_iter_[0]),
            "fit_seconds": time.perf_counter() - started,
            "artifact_sha256": checksum(candidate_path),
            "spline_numeric_columns": structure.spline_columns.tolist()
            if structure is not None
            else [],
            **selection_diagnostics(tune, raw),
        }
        validation_results.append(result)
        write_json(REPORTS / "validation-results.json", validation_results)
        print(
            "Completed",
            config["name"],
            "selection score",
            result["selection_score"],
            flush=True,
        )
        del matrix_fit, matrix_tune, classifier, candidate
        gc.collect()
    selected = max(validation_results, key=lambda item: tuple(item["selection_score"]))
    bundle = joblib.load(ARTIFACTS / (selected["name"] + ".joblib"))
    write_json(
        REPORTS / "selected-before-test.json",
        {
            "selected_at_utc": datetime.now(timezone.utc).isoformat(),
            "selected": selected,
            "spec_sha256": spec_hash,
            "selection_partition": "tune",
            "test_predictions_observed": False,
        },
    )
    del text_fit, text_tune, fit_blocks, tune_blocks
    gc.collect()
display(
    pd.DataFrame(
        [
            {
                k: item[k]
                for k in [
                    "name",
                    "C",
                    "fit_rows",
                    "features",
                    "macro_source_recall",
                    "min_source_recall",
                    "fit_seconds",
                ]
            }
            for item in validation_results
        ]
    )
)
print("Selected solely from development:", selected["name"])

Preparing fit-only feature blocks...


Fitting repeat_v6 on 532751 URLs...


Completed repeat_v6 selection score [0.43670509448135053, 0.4075067024128686, 0.8698285229539164]


Fitting structural_linear_c2 on 532751 URLs...


Completed structural_linear_c2 selection score [0.437732077159365, 0.36193029490616624, 0.8736917928677356]


Fitting structural_spline_c05 on 532751 URLs...


Completed structural_spline_c05 selection score [0.3946261462739419, 0.28418230563002683, 0.87280448721704]


Fitting structural_spline_c2 on 532751 URLs...


Completed structural_spline_c2 selection score [0.4359147684818291, 0.35924932975871315, 0.8743667242296582]


,name,C,fit_rows,features,macro_source_recall,min_source_recall,fit_seconds
0,repeat_v6,2.0,532751,118689,0.436705,0.407507,73.411926
1,structural_linear_c2,2.0,532751,118734,0.437732,0.361930,181.600713
2,structural_spline_c05,0.5,532751,118758,0.394626,0.284182,71.283819
3,structural_spline_c2,2.0,532751,118758,0.435915,0.359249,106.263160


Selected solely from development: structural_linear_c2


## Kalibrasi, threshold konservatif dan pembekuan

Sigmoid memakai calibration saja. Warning menargetkan upper Wilson FPR 1% per sumber bila ukuran benign memungkinkan; sumber kecil memakai batas empiris dan ditandai. High memakai anggaran 0,1% dengan batas yang sama. Kebijakan ini lebih konservatif daripada threshold empiris v6; hasil perbandingan tetap menyertakan perubahan kebijakan. Tidak ada threshold yang dipilih dari test.

In [5]:
if not REUSE_FROZEN:
    with threadpool_limits(limits=4):
        raw_calibration = np.clip(
            raw_probabilities(bundle, calibration.core), 1e-7, 1 - 1e-7
        )
        calibrator = LogisticRegression(
            C=1000, solver="lbfgs", max_iter=500, random_state=SEED
        )
        calibrator.fit(logit(raw_calibration).reshape(-1, 1), calibration.label)
        assert calibrator.coef_[0, 0] > 0
        bundle["calibrator"] = calibrator
        threshold_probability = probabilities(bundle, threshold_data.core)
        old_threshold_probability = old_probabilities(base, threshold_data.core)
    old_control = {
        "warning": source_point(
            threshold_data, old_threshold_probability, 0.01, confidence=True
        ),
        "high": source_point(
            threshold_data, old_threshold_probability, 0.001, confidence=True
        ),
    }
    warning = source_point(threshold_data, threshold_probability, 0.01, confidence=True)
    high = source_point(threshold_data, threshold_probability, 0.001, confidence=True)
    thresholds = {
        "warning": warning["threshold"],
        "high": max(high["threshold"], warning["threshold"]),
    }
    metadata = {
        "version": "notebook-url-core-v7",
        "feature_schema": SCHEMA_VERSION,
        "config": selected,
        "feature_specification": SPECIFICATION,
        "spec_sha256": spec_hash,
        "code_sha256": code_hash,
        "source_hashes": source_hashes,
        "versions": specification["versions"],
        "thresholds": thresholds,
        "high_enabled": thresholds["high"] <= 1,
        "fit_rows": len(fit),
        "fit_sha256": checksum(fit_path),
        "parent_development_sha256": checksum(development_path),
        "new_test_sha256": checksum(external_run_path),
        "threshold_policy": {"warning": warning, "high": high},
        "v6_control_thresholds": {
            level: point["threshold"] for level, point in old_control.items()
        },
        "v6_control_policy": old_control,
        "noncommercial_research_only": True,
        "production_ready": False,
        "calibration_brier_in_sample": float(
            brier_score_loss(
                calibration.label,
                calibrator.predict_proba(logit(raw_calibration).reshape(-1, 1))[:, 1],
            )
        ),
        "calibration_brier_scope": "descriptive_in_sample_not_test",
    }
    bundle["metadata"] = metadata
    joblib.dump(bundle, MODEL, compress=3)
    frozen = {
        "frozen_at_utc": datetime.now(timezone.utc).isoformat(),
        "spec_sha256": spec_hash,
        "code_sha256": code_hash,
        "source_hashes": source_hashes,
        "selected": selected,
        "bundle_sha256": checksum(MODEL),
        "metadata": metadata,
        "test_predictions_observed": False,
    }
    write_json(FROZEN, frozen)
assert checksum(MODEL) == frozen["bundle_sha256"]
assert bundle["metadata"] == frozen["metadata"]
assert notebook_code_hash() == frozen["code_sha256"]
reloaded = joblib.load(MODEL)
probes = [
    "http://example.org/login",
    "https://example.org/login",
    "https://alice.github.io/account",
    "https://example.org/token/REDACTED",
]
with threadpool_limits(limits=4):
    probe_probabilities = probabilities(bundle, probes)
    assert np.array_equal(probe_probabilities, probabilities(reloaded, probes))
    assert probe_probabilities[0] == probe_probabilities[1]
    assert np.isfinite(probe_probabilities).all()
artifact_checks = {
    "save_reload_exact_parity": True,
    "http_https_parity": True,
    "url_qr_same_core_pipeline": True,
    "finite_probability": True,
    "fit_only_transformations": True,
    "thresholds_selected_without_test": True,
    "protected_v3_v6_artifacts_unchanged": all(
        checksum(ROOT / p) == digest for p, digest in protected.items()
    ),
}
assert all(artifact_checks.values())
write_json(REPORTS / "artifact-checks.json", artifact_checks)
print("Frozen model:", bundle["metadata"]["version"], "selected:", selected["name"])
print("Thresholds:", bundle["metadata"]["thresholds"])

Frozen model: notebook-url-core-v7 selected: structural_linear_c2
Thresholds: {'warning': 0.9177038674825727, 'high': 0.9887723682878329}


## Evaluasi sesudah model dan kebijakan dibekukan

Phish360 merupakan sumber uji tambahan yang belum diprediksi saat selection. Domain/URL lama dikeluarkan menggunakan snapshot registry sebelum intake. Dataset v5/v6 dan PhishTank 2026 kini benchmark teramati. Kontrol v6 pada kebijakan threshold v7 yang sama dihitung hanya dari development sebelum test; kebijakan asli v6 tetap dilaporkan.

Angka vendor, kurasi penulis, audit manusia kecil dan feed positif ditampilkan terpisah. Wilson mengasumsikan baris independen; interval bootstrap memakai domain/tenant dan tidak menghapus bias sumber atau campaign.

In [6]:
assert checksum(MODEL) == frozen["bundle_sha256"]
assert notebook_code_hash() == frozen["code_sha256"]
assert checksum(external_run_path) == frozen["metadata"]["new_test_sha256"]
historical_path = (
    ROOT / "ml/runs/notebook-tfidf-logistic-v3/data/development-v3.parquet"
)
historical = pd.read_parquet(historical_path)
datasets = {
    "new_phish360_author_curated": external,
    "observed_v6_deepurlbench_holdout": pd.read_parquet(
        V6 / "data/deepurlbench-holdout-v6.parquet"
    ),
    "observed_v5_human_audit": pd.read_parquet(V5 / "data/phishvn-human-new.parquet"),
    "observed_phresh_novel_domain": historical[
        historical.partition.eq("test") & historical.novel_domain.eq(True)
    ].reset_index(drop=True),
    **{
        "observed_group_holdout_" + source: data.reset_index(drop=True)
        for source, data in historical[
            historical.partition.eq("observed_group_holdout")
        ].groupby("training_source")
    },
    "observed_urlphish": pd.read_parquet(
        ROOT / "ml/runs/notebook-tfidf-logistic-v3/data/urlphish-fresh.parquet"
    ),
    "observed_phishtank_2026_positive": pd.read_parquet(
        V6 / "data/phishtank-2026-v6.parquet"
    ),
}
evaluation, intervals, table = {}, {}, []
policies = {
    "v6_original": base["metadata"]["thresholds"],
    "v6_same_confidence_policy": bundle["metadata"]["v6_control_thresholds"],
    "v7": bundle["metadata"]["thresholds"],
}
with threadpool_limits(limits=4):
    for name, data in datasets.items():
        assert set(data.group).isdisjoint(development.group)
        assert set(data.digest).isdisjoint(development.digest)
        print("Evaluate", name, len(data), flush=True)
        old_p = old_probabilities(base, data.core)
        new_p = probabilities(bundle, data.core)
        evaluation[name] = {
            "rows": len(data),
            "groups": int(data.group.nunique()),
            "new_independent_source": name.startswith("new_phish360"),
            "data_sha256": checksum(external_run_path)
            if name.startswith("new_phish360")
            else None,
            "models": {},
        }
        for model_name, policy in policies.items():
            p = new_p if model_name == "v7" else old_p
            evaluation[name]["models"][model_name] = {}
            for level, threshold in policy.items():
                item = safe_metrics(data.label, p, threshold)
                evaluation[name]["models"][model_name][level] = item
                table.append(
                    {
                        "dataset": name,
                        "model": model_name,
                        "policy": level,
                        "rows": len(data),
                        "groups": int(data.group.nunique()),
                        **{
                            key + "_pct": None if item[key] is None else 100 * item[key]
                            for key in ["accuracy", "recall", "fpr", "precision"]
                        },
                    }
                )
        intervals[name] = {
            reference: paired_domain_interval(
                data,
                old_p,
                new_p,
                policies[reference]["warning"],
                policies["v7"]["warning"],
                repeats=1000,
            )
            for reference in ["v6_original", "v6_same_confidence_policy"]
        }
evaluation_report = {
    "evaluated_at_utc": datetime.now(timezone.utc).isoformat(),
    "bundle_sha256": frozen["bundle_sha256"],
    "code_sha256": code_hash,
    "spec_sha256": spec_hash,
    "new_test_manifest_sha256": checksum(REPORTS / "frozen-new-test-manifest.json"),
    "new_test_used_for_selection_fitting_calibration_threshold": False,
    "benchmark_results": evaluation,
    "paired_domain_intervals": intervals,
}
write_json(REPORTS / "evaluation-v7.json", evaluation_report)
results_table = pd.DataFrame(table)
display(results_table[results_table.policy.eq("warning")].round(4))
print("Positive-only FPR/accuracy/traffic precision are unavailable, not zero.")

Evaluate new_phish360_author_curated 7516


Evaluate observed_v6_deepurlbench_holdout 20000


Evaluate observed_v5_human_audit 33


Evaluate

 observed_phresh_novel_domain 80052


Evaluate observed_group_holdout_mendeley2020 1002


... (2 outputs left out)

Evaluate observed_urlphish 85824


Evaluate observed_phishtank_2026_positive 9269


,dataset,model,policy,rows,groups,accuracy_pct,recall_pct,fpr_pct,precision_pct
0,new_phish360_author_curated,v6_original,warning,7516,7002,75.6519,42.9866,0.1159,99.6379
2,new_phish360_author_curated,v6_same_confidence_policy,warning,7516,7002,72.9377,36.6136,0.1159,99.5752
4,new_phish360_author_curated,v7,warning,7516,7002,75.1464,41.8932,0.1854,99.4070
6,observed_v6_deepurlbench_holdout,v6_original,warning,20000,19139,93.2250,46.3139,0.9444,85.9060
8,observed_v6_deepurlbench_holdout,v6_same_confidence_policy,warning,20000,19139,92.7150,39.3487,0.6521,88.2353
10,observed_v6_deepurlbench_holdout,v7,warning,20000,19139,93.0950,43.1027,0.6914,88.5688
12,observed_v5_human_audit,v6_original,warning,33,33,45.4545,18.1818,0.0000,100.0000
14,observed_v5_human_audit,v6_same_confidence_policy,warning,33,33,42.4242,13.6364,0.0000,100.0000
16,observed_v5_human_audit,v7,warning,33,33,42.4242,13.6364,0.0000,100.0000
18,observed_phresh_novel_domain,v6_original,warning,80052,59880,63.7548,43.0951,0.1613,99.7862


Positive-only FPR/accuracy/traffic precision are unavailable, not zero.


## Grafik, diagnosis dan keputusan

Grafik membedakan test sumber baru dan benchmark teramati. Pengujian kode tidak menggantikan recall/FPR. Artifact v7 disimpan sebagai kandidat riset; loader/API shadow v6 tetap tidak berubah. Perbaikan atau regresi yang tampak pada test tidak digunakan untuk mengubah konfigurasi v7.

In [7]:
with threadpool_limits(limits=4):
    tune_old = old_probabilities(base, tune.core)
    tune_new = probabilities(bundle, tune.core)
from guardian_ml.preprocessing import EXTRACT

parsed_tune = [prepare_url(core) for core in tune.core]
probe_slices = {
    "private_hosting": np.array([EXTRACT(p.host).is_private for p in parsed_tune]),
    "punycode": np.array(["xn--" in p.host for p in parsed_tune]),
    "url_over_120_chars": np.array(
        [len(p.core.split("://", 1)[1]) > 120 for p in parsed_tune]
    ),
    "redacted_placeholder": np.array(["REDACTED" in p.path for p in parsed_tune]),
}
slice_report = {
    name: {
        "rows": int(mask.sum()),
        "v6_original": safe_metrics(
            tune.label[mask], tune_old[mask], policies["v6_original"]["warning"]
        ),
        "v7": safe_metrics(tune.label[mask], tune_new[mask], policies["v7"]["warning"]),
    }
    for name, mask in probe_slices.items()
}
selected_phish360 = evaluation["new_phish360_author_curated"]["models"]["v7"]["warning"]
selected_vendor = evaluation["observed_v6_deepurlbench_holdout"]["models"]["v7"][
    "warning"
]
old_vendor = evaluation["observed_v6_deepurlbench_holdout"]["models"]["v6_original"][
    "warning"
]
quality = {
    "indonesia_human_gold_available": False,
    "production_ready": False,
    "phish360_recall_lower95_at_least_85pct": selected_phish360[
        "recall_wilson95_rows_iid_assumption"
    ][0]
    >= 0.85,
    "phish360_fpr_upper95_at_most_1pct": selected_phish360[
        "fpr_wilson95_rows_iid_assumption"
    ][1]
    <= 0.01,
    "vendor_recall_increased_vs_v6_original": selected_vendor["recall"]
    > old_vendor["recall"],
    "vendor_fpr_upper95_at_most_1pct": selected_vendor[
        "fpr_wilson95_rows_iid_assumption"
    ][1]
    <= 0.01,
    "public_application_activated": False,
    "runtime_v6_preserved": True,
}
assert all(checksum(ROOT / path) == digest for path, digest in protected.items())
assert notebook_code_hash() == frozen["code_sha256"]
assert checksum(MODEL) == frozen["bundle_sha256"]
summary = {
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "protocol": "ai-agent-docs/structural-retraining-v7-plan.md",
    "notebook": NOTEBOOK.relative_to(ROOT).as_posix(),
    "experiment": "notebook-tfidf-logistic-v7",
    "classifier": "LogisticRegression",
    "development_audit": fit_audit,
    "feature_specification": SPECIFICATION,
    "validation_results": validation_results,
    "selected_before_test": selected,
    "frozen_at_utc": frozen["frozen_at_utc"],
    "bundle_sha256": frozen["bundle_sha256"],
    "code_sha256": code_hash,
    "spec_sha256": spec_hash,
    "thresholds": bundle["metadata"]["thresholds"],
    "v6_control_thresholds": bundle["metadata"]["v6_control_thresholds"],
    "new_source_audit": test_manifest,
    **evaluation_report,
    "development_structural_slices": slice_report,
    "artifact_checks": artifact_checks,
    "quality_status": quality,
    "protected_baseline_checksums": protected,
    "noncommercial_research_only": True,
    "limitations": [
        "Phish360 labels are author-curated, not two independently published per-case human judgments.",
        "New source is historical and not an Indonesian gold set; timestamps/campaign independence remain unverified.",
        "Text encoders retain unsupervised exposure to original v6 fit, including quarantined rows.",
        "Threshold policy differs from original v6; same-policy control is reported separately.",
        "Old benchmarks are observed and were not used to select v7.",
        "No automatic relabel, target visit, new human annotation, public activation, or guarantee from LOW scores.",
    ],
}
write_json(ROOT / "ml/reports/structural-retraining-v7.json", summary)
write_json(REPORTS / "summary-v7.json", summary)
warning_table = results_table[
    results_table.policy.eq("warning") & results_table.model.isin(["v6_original", "v7"])
]
names = [
    "new_phish360_author_curated",
    "observed_v6_deepurlbench_holdout",
    "observed_phishtank_2026_positive",
]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for axis, metric in zip(axes, ["recall_pct", "fpr_pct"]):
    for offset, model, color in [
        (-0.18, "v6_original", "#64748b"),
        (0.18, "v7", "#0f766e"),
    ]:
        values = [
            warning_table[
                (warning_table.dataset == name) & (warning_table.model == model)
            ][metric].iloc[0]
            for name in names
        ]
        axis.bar(
            np.arange(len(names)) + offset,
            [np.nan if pd.isna(v) else v for v in values],
            width=0.36,
            label=model,
            color=color,
        )
    axis.set_xticks(
        range(len(names)),
        ["Phish360 (new)", "Vendor (observed)", "PhishTank (observed)"],
        rotation=15,
    )
    axis.set_ylabel(metric.replace("_pct", "") + " (%)")
    axis.legend()
fig.tight_layout()
fig.savefig(REPORTS / "evaluation-v7.png", dpi=150)
display(fig)
plt.close(fig)
display(pd.DataFrame(quality, index=["status"]).T)
print("Candidate research model saved; no test-based retuning or production promotion.")

<Figure size 1200x450 with 2 Axes>

,status
indonesia_human_gold_available,False
production_ready,False
phish360_recall_lower95_at_least_85pct,False
phish360_fpr_upper95_at_most_1pct,True
vendor_recall_increased_vs_v6_original,False
vendor_fpr_upper95_at_most_1pct,True
public_application_activated,False
runtime_v6_preserved,True


Candidate research model saved; no test-based retuning or production promotion.
